
# Pipeline overview 

- Load **X** (genes) and **Y** (proteins); pick **one protein** to predict.
- Take a **tiny subset** of samples/genes to keep it fast.
- Do **leave-one-out (LOO)**: for each sample, hold it out as test.
- On the remaining training set, run **inner K-fold CV** to pick XGBoost hyperparams.
- Within each LOO iteration, do **feature selection**:
- Train XGBoost on the selected features for each param combo; pick the one with **lowest MSE**.
- **Recompute feature selection** on full train with the chosen threshold; **refit** XGBoost and **predict** the held-out sample.
- Collect a row per test sample: **true value**, **predicted value**, **chosen params**, and (if available) **feature importances**.
- Finally, show a quick **RMSE** over the LOO predictions.



In [ ]:
#SETUP (paths, deps, imports)
import os, sys, subprocess, importlib
from pathlib import Path

# Detect project root (works if notebook is at project root or inside it)
CWD = Path.cwd()
if (CWD / "src" / "pipeline_functions.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "src" / "pipeline_functions.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    # last resort: assume repo name
    maybe = [p for p in CWD.parents if (p / "src" / "pipeline_functions.py").exists()]
    PROJECT_ROOT = maybe[0] if maybe else CWD

SRC_DIR  = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path so we can import your code
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Light dep check (install if missing for local notebooks)
def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        pip_name = pip_name or pkg
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pkg, pipn in [
    ("joblib","joblib"),
    ("pandas","pandas"),
    ("numpy","numpy"),
    ("sklearn","scikit-learn"),
    ("xgboost","xgboost>=2.0"),
    ("openpyxl","openpyxl>=3.1"),
]:
    _ensure(pkg, pipn)

import numpy as np, pandas as pd
from joblib import dump, load
import xgboost as xgb

from sklearn.model_selection import KFold, LeaveOneOut, RandomizedSearchCV
from src.pipeline_functions import cv_pipeline_loo
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


print("Project root:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


Project root: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb
SRC_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\src
DATA_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data


In [45]:
#SYNTHETIC DATA GENERATION
#Small and quick: ~30 samples, 300 features, 3 proteins
rng = np.random.default_rng(123)
n_samples, n_features, n_proteins = 30, 300, 3

X_np = rng.normal(size=(n_samples, n_features))
# make P0 depend on a sparse subset of features + noise; P1 similar; P2 mostly noise
w0 = np.zeros(n_features); w0[:15] = rng.normal(0, 1, size=15)
w1 = np.zeros(n_features); w1[50:70] = rng.normal(0, 1, size=20)

y0 = X_np @ w0 + rng.normal(0, 0.5, size=n_samples)
y1 = X_np @ w1 + rng.normal(0, 0.6, size=n_samples)
y2 = rng.normal(0, 1.0, size=n_samples)

samples  = [f"S{i:02d}" for i in range(n_samples)]
features = [f"G{j:04d}" for j in range(n_features)]
proteins = [f"P{k}" for k in range(n_proteins)]

X = pd.DataFrame(X_np, index=samples, columns=features)
Y = pd.DataFrame({"P0": y0, "P1": y1, "P2": y2}, index=samples)

# Save exactly with the filenames your pipeline expects
x_path = DATA_DIR / "transcriptomics_synthetic.joblib"
y_path = DATA_DIR / "proteomics_synthetic.joblib"
dump(X, x_path); dump(Y, y_path)

display(X.shape, Y.shape, x_path, y_path)


(30, 300)

(30, 3)

WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/pipeline_xgb/data/transcriptomics_synthetic.joblib')

WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/pipeline_xgb/data/proteomics_synthetic.joblib')

In [46]:
# REAL DATA:
X = load(DATA_DIR / "transcriptomics_filtered.joblib")
Y = load(DATA_DIR / "proteomics_matched.joblib")

In [ ]:
#SANITY CHECKS
if "Unnamed: 0" in X.columns:
    X = X.set_index("Unnamed: 0")

# Clean labels and set nice axis names
X.index = X.index.astype(str).str.strip()
X.columns = X.columns.astype(str).str.strip()
X.index.name, X.columns.name = "samples", "genes"

# Ensure numeric values (bad entries → NaN; also clear infs)
X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
Y.index = Y.index.astype(str).str.strip()
Y.columns = Y.columns.astype(str).str.strip()
Y.index.name, Y.columns.name = "samples", "proteins"
Y = Y.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)



In [48]:
X

genes,A1BG,A1BG-AS1,A1CF,A2M,A2ML1,A4GALT,AAAS,AACS,AACSP1,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
samples,,,,,,,,,,,,,,,,,,,,,
MDAMB468_BREAST,2.689299,2.954196,0.014355,3.329124,0.443607,1.575312,5.311430,4.531069,0.056584,0.124328,...,4.139142,6.986752,2.192194,3.114367,4.250962,2.792855,3.250962,6.233811,3.512227,5.221490
SH4_SKIN,4.980025,3.583760,0.000000,8.405099,0.000000,0.111031,5.229972,4.095924,0.070389,0.014355,...,3.698218,6.113742,1.384050,2.659925,4.979111,0.014355,2.553361,6.644000,3.659925,4.249445
AU565_BREAST,2.518535,2.269033,0.014355,1.275007,2.451541,0.933573,6.011898,5.564378,1.618239,0.250962,...,4.235727,7.373909,0.632268,2.049631,3.471187,0.014355,2.794936,7.776828,3.814550,3.978196
KMRC1_KIDNEY,0.084064,0.070389,0.000000,0.137504,0.000000,2.903038,4.842476,3.357552,0.056584,0.124328,...,4.584361,6.704734,0.432959,1.021480,1.475085,0.485427,1.384050,5.510962,1.263034,3.416840
CAL51_BREAST,4.729553,2.807355,0.014355,0.575312,0.713696,3.751678,6.262847,3.874797,0.056584,0.070389,...,4.835924,7.356496,1.700440,2.424922,3.884598,0.084064,3.558268,6.812113,4.068671,4.939697
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FADU_UPPER_AERODIGESTIVE_TRACT,0.070389,0.389567,0.028569,0.028569,0.263034,4.168321,5.244887,5.409391,0.189034,2.446256,...,4.865919,7.543882,0.014355,0.028569,3.807355,0.790772,3.247928,5.279471,4.277241,5.158660
KP4_PANCREAS,0.389567,0.632268,0.042644,0.367371,0.150560,2.163499,6.280585,4.335569,0.014355,4.980939,...,5.210233,6.656067,2.137504,2.558268,3.754888,0.042644,2.912650,6.915401,3.799087,4.475085
MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,3.945795,2.185867,0.014355,0.000000,0.070389,0.042644,5.667608,3.218781,0.056584,0.000000,...,4.143230,7.929318,1.137504,1.835924,4.955127,0.056584,2.107688,5.816856,3.693766,5.030777


In [49]:
Y

proteins,A2M,AAAS,AACS,AAGAB,AAK1,AAMDC,AAMP,AAR2,AARS,AARS2,...,ZNF787,ZNF800,ZNF830,ZNHIT6,ZNRD1,ZRANB2,ZW10,ZWILCH,ZYX,ZZZ3
samples,,,,,,,,,,,,,,,,,,,,,
MDAMB468_BREAST,-0.345562,-0.109405,0.370945,0.009639,0.005800,1.588762,-0.364119,-0.031192,-0.052310,-0.282176,...,-0.094039,-0.273452,-0.222986,0.110619,-0.220072,0.320284,-0.109837,-0.234132,-1.032888,-0.054057
SH4_SKIN,1.642186,-0.283335,0.162102,-0.057517,0.760639,0.119699,0.105279,-0.075316,0.399385,1.102265,...,-0.248390,0.531159,-0.452978,-0.386262,-0.047313,-0.499179,-0.208351,-0.163901,-0.361225,-0.371537
AU565_BREAST,-1.129114,0.024663,1.071283,-0.758048,-0.320451,-1.068118,-0.158266,0.365150,-0.797109,-0.272314,...,0.043493,1.607322,-0.547892,-0.175286,0.061356,-0.175592,0.071945,-0.393799,0.309026,-0.301741
KMRC1_KIDNEY,1.493333,-0.588886,0.404605,-0.210457,0.533949,0.437453,-0.070486,-0.045454,0.048472,-0.352636,...,-0.226766,-0.363713,0.001180,-0.517588,-0.286447,-0.326172,0.019450,-0.055173,0.574568,-0.732279
CAL51_BREAST,0.274652,0.673324,-0.587890,-0.044001,-0.778679,0.051707,0.294878,-0.211209,-0.436950,0.113565,...,0.319720,-0.184053,0.322673,-0.294126,0.161771,-0.060968,0.103745,0.151656,-0.218916,0.697868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FADU_UPPER_AERODIGESTIVE_TRACT,-1.116991,-0.779655,-0.343373,0.032533,-1.433436,-0.991085,-0.899321,-0.620034,-0.625103,-0.697012,...,0.821462,0.343556,0.993749,-0.348426,1.067482,0.125648,-0.894195,-0.610845,-0.306374,-0.005547
KP4_PANCREAS,1.922976,-0.628770,-1.200965,2.101721,-0.549493,1.602728,0.949461,-0.512347,-0.856384,-0.896097,...,-0.473678,-0.024037,1.098143,0.337931,1.372841,-0.052092,-0.519440,-0.873276,1.281937,1.008579
MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,-1.106344,1.205109,0.660346,-0.539838,0.951283,0.139173,-0.211486,0.418099,-0.894781,0.713249,...,0.271313,0.084621,0.244403,0.347557,0.136414,0.282679,0.672338,0.885732,-0.498927,-0.157715


In [ ]:
#Direct function test (very small grid; CPU hist) 
use_gpu = False  # True -> gpu_hist + gpu_predictor

param_grid = {
    # capacity / shape
    "max_depth": [3, 6, 8],
    "min_child_weight": [1, 3],          # stronger when larger
    "gamma": [0.0, 0.1],                  # minimum loss reduction to split
    "learning_rate": [0.05, 0.1],
    "n_estimators": [600, 1200, 2000],    # early stopping will clip this

    # sampling
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.6, 0.8, 1.0],

    # regularization
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [0.5, 1.0, 2.0],

    # histogram params
    "max_bin": [256],

    # device
    "tree_method": ["gpu_hist"] if use_gpu else ["hist"],
    **({"predictor": ["gpu_predictor"]} if use_gpu else {}),
}

protein_name = "A2M"
res_df = cv_pipeline_loo(
    model=xgb.XGBRegressor(objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0),
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    feature_selector="spearman", 
    n_splits=5,                   # inner CV folds (tiny for speed)
    select_k=500,                  # top-k by p-value inside folds
    early_stopping_rounds=10,
)

display(res_df.head())
print("Rows (should equal #samples):", len(res_df))

# Quick metrics on the LOO predictions

mse = mean_squared_error(res_df["true"], res_df["predicted"])
r, p = pearsonr(res_df["true"], res_df["predicted"])
print(f"MSE: {mse:.4f} | Pearson r: {r:.3f} (p={p:.1e})")
assert res_df["predicted"].notna().all(), "NaNs in predictions!"
print("✅ Function pipeline smoke test passed.")


c:\Users\hp\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,features_selected
0,A2M,MDAMB468_BREAST,-0.345562,-0.766416,"{'max_depth': 3, 'min_child_weight': 1, 'gamma...","{'HSH2D': 0.1009085401892662, 'SCARF2': 8.0577...","[HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA..."
1,A2M,SH4_SKIN,1.642186,0.187917,"{'max_depth': 3, 'min_child_weight': 1, 'gamma...","{'HSH2D': 0.1874517947435379, 'SCARF2': 4.8841...","[HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA..."
2,A2M,AU565_BREAST,-1.129114,-0.407232,"{'max_depth': 3, 'min_child_weight': 1, 'gamma...","{'HSH2D': 0.04900778457522392, 'SCARF2': 6.018...","[HSH2D, SCARF2, CNPY4, SYDE1, LRP1, CALD1, SPA..."
3,A2M,KMRC1_KIDNEY,1.493333,0.648681,"{'max_depth': 3, 'min_child_weight': 1, 'gamma...","{'SCARF2': 2.57433819770813, 'HSH2D': 0.187106...","[SCARF2, HSH2D, CNPY4, SYDE1, LRP1, CALD1, ENA..."
4,A2M,CAL51_BREAST,0.274652,0.349149,"{'max_depth': 3, 'min_child_weight': 1, 'gamma...","{'HSH2D': 0.05797899514436722, 'SCARF2': 6.096...","[HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, ARH..."


Rows (should equal #samples): 369
MSE: 1.0764 | Pearson r: 0.535 (p=1.1e-28)
✅ Function pipeline smoke test passed.


In [51]:
res_df['features_selected']

0      [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
1      [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
2      [HSH2D, SCARF2, CNPY4, SYDE1, LRP1, CALD1, SPA...
3      [SCARF2, HSH2D, CNPY4, SYDE1, LRP1, CALD1, ENA...
4      [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, ARH...
                             ...                        
364    [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
365    [SCARF2, HSH2D, CNPY4, CALD1, SYDE1, LRP1, SPA...
366    [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
367    [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
368    [HSH2D, SCARF2, CNPY4, LRP1, SYDE1, CALD1, SPA...
Name: features_selected, Length: 369, dtype: object

In [52]:
# feature similarity analysis
from collections import Counter

# Turn lists -> sets
feat_sets = [set(fs) for fs in res_df['features_selected']]
n_splits = len(feat_sets)
k = len(next(iter(feat_sets)))  # features per split (assumes constant select_k)

# (a) Core & frequency
core = set.intersection(*feat_sets)              # present in ALL splits
freq = Counter(f for s in feat_sets for f in s)  # how often each feature appears
freq_df = (pd.DataFrame.from_dict(freq, orient="index", columns=["count"])
           .sort_values("count", ascending=False))
freq_df["freq"] = freq_df["count"] / n_splits

print(f"Splits: {n_splits}, features per split: {k}")
print(f"Features in EVERY split (core): {len(core)}")
print(f"Features appearing in ≥80% of splits: {(freq_df['freq']>=0.8).sum()}")

# (b) Pairwise overlap & Jaccard (how similar sets are on average)
def jacc(a,b):
    inter = len(a & b); uni = len(a | b)
    return inter / uni if uni else 1.0

overlaps = []
jaccs = []
for i in range(n_splits):
    for j in range(i+1, n_splits):
        inter = len(feat_sets[i] & feat_sets[j])
        overlaps.append(inter)
        jaccs.append(jacc(feat_sets[i], feat_sets[j]))

avg_same   = float(np.mean(overlaps))
avg_change = k - avg_same
avg_jacc   = float(np.mean(jaccs))
print(f"Avg same features between two splits: {avg_same:.1f}")
print(f"Avg changed features between two splits: {avg_change:.1f}")
print(f"Avg Jaccard similarity: {avg_jacc:.3f}")

# (c) Per-split agreement vs a consensus set of size k (top-k most frequent)
consensus_k = set(freq_df.head(k).index)
res_df["overlap_with_consensus"] = [len(s & consensus_k) for s in feat_sets]
res_df["changed_vs_consensus"]   = k - res_df["overlap_with_consensus"]
res_df[["LOO sample","overlap_with_consensus","changed_vs_consensus"]].head()

Splits: 369, features per split: 500
Features in EVERY split (core): 436
Features appearing in ≥80% of splits: 482
Avg same features between two splits: 486.1
Avg changed features between two splits: 13.9
Avg Jaccard similarity: 0.946


,LOO sample,overlap_with_consensus,changed_vs_consensus
0,MDAMB468_BREAST,497,3
1,SH4_SKIN,487,13
2,AU565_BREAST,491,9
3,KMRC1_KIDNEY,484,16
4,CAL51_BREAST,490,10


In [ ]:
# CLI end-to-end test for pipeline.py
import sys, subprocess
from pathlib import Path
from joblib import load
from IPython.display import display
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# Locate pipeline.py
CWD = Path.cwd().resolve()
candidates = [CWD / "src" / "pipeline.py", CWD / "pipeline.py"]
pipeline_path = next((p for p in candidates if p.exists()), None)
if pipeline_path is None:
    found = list(CWD.rglob("pipeline.py"))
    if not found:
        raise FileNotFoundError("Could not find pipeline.py under ./src or project root.")
    pipeline_path = found[0]

SRC_DIR = pipeline_path.parent
DATA_DIR = SRC_DIR.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Build CLI for the new pipeline (CPU run; add "--force_gpu" to use GPU)
cli = [
    sys.executable,
    str(pipeline_path),
    "--protein", "A2M",
    "--splits", "5",
    "--select_k", "500",        # Spearman prefilter size per LOO split (0 = keep all)
    "--early_stopping", "10",
    "--max_param_combos", "12",   # optional: sample the param grid
    # "--minimal_save",              # optional
    # "--force_gpu",                 # optional: use GPU if available
    # "--no_feature_selection",      # optional: equivalent to --select_k 0
]

print("Running:", " ".join(cli))
ret = subprocess.run(cli, cwd=str(SRC_DIR), capture_output=True, text=True)
print("Return code:", ret.returncode)
print("--- STDOUT ---")
print(ret.stdout[-2000:] if ret.stdout else "")
print("--- STDERR ---")
print(ret.stderr[-2000:] if ret.stderr else "")

if ret.returncode != 0:
    raise RuntimeError("CLI run failed. See STDERR above.")



Running: c:\Users\hp\anaconda3\python.exe C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\src\pipeline.py --protein A2M --splits 5 --select_k 500 --early_stopping 10 --max_param_combos 12
Return code: 0
--- STDOUT ---
Loading data...
Running loo for protein 'A2M' (outer LOO across samples)...
Saving: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data\A2M_loo_xgboost_LOO_results.joblib
Done.

--- STDERR ---
c:\Users\hp\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



In [ ]:
# Load the results saved by pipeline.py
out_path = DATA_DIR / "A2M_loo_xgboost_LOO_results.joblib"
if not out_path.exists():
    raise FileNotFoundError(f"Expected results file not found: {out_path}")

cli_df = load(out_path)
display(cli_df.head(), out_path)

# Quick metrics
mse = mean_squared_error(cli_df["true"], cli_df["predicted"])
r, p = pearsonr(cli_df["true"], cli_df["predicted"])
print(f"[CLI] MSE: {mse:.4f} | Pearson r: {r:.3f} (p={p:.1e})")
print("✅ CLI pipeline smoke test passed.")

FileNotFoundError: Expected results file not found: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data\A2M_loo_xgboost_LOO_results.joblib